# SEER GBM 生存分析

基于清洗后的 SEER 数据，进行 Kaplan-Meier 生存分析和 Log-rank 检验。

分析变量:
- Survival months: 生存时间 (月)，来自 SEER SRV_TIME_MON
- Vital status recode: 终点事件 (0=删失, 1=死亡)
- 分层变量: Age Group, Sex, Race, Grade, Stage, Surgery, Radiation, Chemotherapy

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

# 全局绘图设置
mpl.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.linewidth': 0.8,
    'axes.unicode_minus': False,
    'figure.dpi': 300,
    'savefig.dpi': 300,
    'savefig.format': 'tiff',
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
})

COLORS = ['#5B9BD5', '#ED7D31', '#70AD47', '#FFC000', '#9B59B6', '#44546A']

df = pd.read_csv('cleaned_data.csv')

In [ ]:
# ===== KM曲线: 年龄分组 =====
# Age Group: <60 / 60-69 / ≥70
# 分组依据: 文献报道年龄是GBM最重要的独立预后因素
fig, ax = plt.subplots(figsize=(7, 5))
kmf = KaplanMeierFitter()
age_groups = ['<60', '60-69', '≥70']

for i, group in enumerate(age_groups):
    mask = df['Age Group'] == group
    kmf.fit(df.loc[mask, 'Survival months'], 
            df.loc[mask, 'Vital status recode'], 
            label=group)
    kmf.plot_survival_function(ax=ax, color=COLORS[i], linewidth=1.5)

ax.set_xlabel('Months since Diagnosis')
ax.set_ylabel('Overall Survival Probability')
ax.set_title('Kaplan-Meier Curves by Age Group', fontweight='bold', pad=10)
ax.legend(title='Age Group', frameon=False)

# Log-rank 检验: 两两比较年龄组间生存差异
# <60 vs 60-69
group1 = df[df['Age Group'] == '<60']
group2 = df[df['Age Group'] == '60-69']
result = logrank_test(group1['Survival months'], group2['Survival months'],
                      group1['Vital status recode'], group2['Vital status recode'])
print(f'Log-rank (<60 vs 60-69): p = {result.p_value:.4e}')

# 60-69 vs ≥70
group3 = df[df['Age Group'] == '≥70']
result2 = logrank_test(group2['Survival months'], group3['Survival months'],
                       group2['Vital status recode'], group3['Vital status recode'])
print(f'Log-rank (60-69 vs ≥70): p = {result2.p_value:.4e}')

plt.tight_layout()
fig.savefig('fig6_km_age.tiff')
plt.close()

In [ ]:
# ===== KM曲线: 性别 =====
# Sex: 0=male, 1=female
fig, ax = plt.subplots(figsize=(7, 5))
kmf = KaplanMeierFitter()
sex_labels = {0: 'Male', 1: 'Female'}

for i, sex in enumerate([0, 1]):
    mask = df['Sex'] == sex
    kmf.fit(df.loc[mask, 'Survival months'], 
            df.loc[mask, 'Vital status recode'], 
            label=sex_labels[sex])
    kmf.plot_survival_function(ax=ax, color=COLORS[i], linewidth=1.5)

ax.set_xlabel('Months since Diagnosis')
ax.set_ylabel('Overall Survival Probability')
ax.set_title('Kaplan-Meier Curves by Sex', fontweight='bold', pad=10)
ax.legend(title='Sex', frameon=False)

# Log-rank 检验
group1 = df[df['Sex'] == 0]
group2 = df[df['Sex'] == 1]
result = logrank_test(group1['Survival months'], group2['Survival months'],
                      group1['Vital status recode'], group2['Vital status recode'])
print(f'Log-rank (Male vs Female): p = {result.p_value:.4e}')

plt.tight_layout()
fig.savefig('fig7_km_sex.tiff')
plt.close()

In [ ]:
# ===== KM曲线: 种族 =====
# Race: White / Black / Asian or Pacific Islander / American Indian/Alaska Native
fig, ax = plt.subplots(figsize=(7, 5))
kmf = KaplanMeierFitter()
race_groups = df['Race'].value_counts().index.tolist()

for i, race in enumerate(race_groups):
    mask = df['Race'] == race
    kmf.fit(df.loc[mask, 'Survival months'], 
            df.loc[mask, 'Vital status recode'], 
            label=race)
    kmf.plot_survival_function(ax=ax, color=COLORS[i % len(COLORS)], linewidth=1.5)

ax.set_xlabel('Months since Diagnosis')
ax.set_ylabel('Overall Survival Probability')
ax.set_title('Kaplan-Meier Curves by Race', fontweight='bold', pad=10)
ax.legend(title='Race', frameon=False, fontsize=9)

plt.tight_layout()
fig.savefig('fig8_km_race.tiff')
plt.close()

In [ ]:
# ===== KM曲线: 分化程度 =====
# Grade Recode: Grade I~IV + Unknown
# Note: 大部分GBM为Grade IV (Undifferentiated/anaplastic)
fig, ax = plt.subplots(figsize=(7, 5))
kmf = KaplanMeierFitter()
grade_groups = ['Well differentiated; Grade I', 'Moderately differentiated; Grade II',
                'Poorly differentiated; Grade III', 'Undifferentiated; anaplastic; Grade IV',
                'Unknown']
grade_labels = ['Grade I', 'Grade II', 'Grade III', 'Grade IV', 'Unknown']

for i, (grade, label) in enumerate(zip(grade_groups, grade_labels)):
    mask = df['Grade'] == grade
    if mask.sum() > 0:
        kmf.fit(df.loc[mask, 'Survival months'], 
                df.loc[mask, 'Vital status recode'], 
                label=label)
        kmf.plot_survival_function(ax=ax, color=COLORS[i % len(COLORS)], linewidth=1.5)

ax.set_xlabel('Months since Diagnosis')
ax.set_ylabel('Overall Survival Probability')
ax.set_title('Kaplan-Meier Curves by Grade', fontweight='bold', pad=10)
ax.legend(title='Grade', frameon=False, fontsize=9)

plt.tight_layout()
fig.savefig('fig9_km_grade.tiff')
plt.close()

In [ ]:
# ===== KM曲线: 分期 =====
# Summary stage 2000: Localized / Regional / Distant / Unknown/unstaged
# Note: GBM以Localized为主，Distant (远处转移) 罕见
fig, ax = plt.subplots(figsize=(7, 5))
kmf = KaplanMeierFitter()
stage_groups = ['Localized', 'Regional', 'Distant', 'Unknown/unstaged']

for i, stage in enumerate(stage_groups):
    mask = df['Stage'] == stage
    if mask.sum() > 0:
        kmf.fit(df.loc[mask, 'Survival months'], 
                df.loc[mask, 'Vital status recode'], 
                label=stage)
        kmf.plot_survival_function(ax=ax, color=COLORS[i % len(COLORS)], linewidth=1.5)

ax.set_xlabel('Months since Diagnosis')
ax.set_ylabel('Overall Survival Probability')
ax.set_title('Kaplan-Meier Curves by Stage', fontweight='bold', pad=10)
ax.legend(title='Stage', frameon=False, fontsize=9)

plt.tight_layout()
fig.savefig('fig10_km_stage.tiff')
plt.close()

In [ ]:
# ===== KM曲线: 治疗方式 (手术/放疗/化疗) =====
# Surgery: 0=No surgery, 1=手术
# Radiation: 0=No radiation, 1=放疗 (含所有放疗方式)
# Chemotherapy: 0=No/Unknown, 1=化疗
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
treatment_vars = ['Surgery', 'Radiation', 'Chemotherapy']
treatment_labels = [
    {0: 'No Surgery', 1: 'Surgery'},
    {0: 'No Radiation', 1: 'Radiation'},
    {0: 'No Chemotherapy', 1: 'Chemotherapy'}
]

for idx, (var, labels) in enumerate(zip(treatment_vars, treatment_labels)):
    ax = axes[idx]
    kmf = KaplanMeierFitter()
    
    for i, (val, label) in enumerate(labels.items()):
        mask = df[var] == val
        kmf.fit(df.loc[mask, 'Survival months'], 
                df.loc[mask, 'Vital status recode'], 
                label=label)
        kmf.plot_survival_function(ax=ax, color=COLORS[i], linewidth=1.5)
    
    ax.set_xlabel('Months since Diagnosis')
    ax.set_ylabel('Overall Survival Probability')
    ax.set_title(f'KM Curves by {var}', fontweight='bold', pad=10)
    ax.legend(title=var, frameon=False, fontsize=9)

plt.tight_layout()
fig.savefig('fig11_km_treatment.tiff')
plt.close()